# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. It loads the dataset defined by a Croissant schema, reviews metadata and available record sets, extracts and analyzes the records, and visualizes results.

### Dataset Source
- Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Install mlcroissant if not available
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the Croissant dataset using `mlcroissant`. This will also provide a quick summary of the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata
print(f"Name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Authors: {dataset.metadata.author}\n")
print(f"Keywords: {dataset.metadata.keywords}\n")
print(f"Version: {dataset.metadata.version}, Published: {dataset.metadata.datePublished}")

## 2. Data Overview

List available record sets, their fields, and corresponding `@id`s. This provides a map of the entities you can load and analyze. All references to data (record sets, fields) are by their `@id`.

In [ ]:
# List all available RecordSets and their fields by @id
record_set_ids = []

print("Available RecordSets in this dataset:")
if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    for rs in dataset.metadata.recordSet:
        print(f"- RecordSet name: {getattr(rs, 'name', 'N/A')}, @id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', 'N/A')}")
        rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None)
        if rs_id:
            record_set_ids.append(rs_id)
else:
    print("No explicit RecordSets listed in Croissant. Attempting to find files from distribution...\n")
    # Fallback: get distributions
    record_set_ids = []
    if hasattr(dataset.metadata, 'distribution') and dataset.metadata.distribution:
        for dist in dataset.metadata.distribution:
            dist_id = dist['@id'] if isinstance(dist, dict) and '@id' in dist else getattr(dist, '@id', None)
            if dist_id:
                print(f"- Distribution/FileObject @id: {dist_id}")
                record_set_ids.append(dist_id)

# For each RecordSet, preview available fields
for idx, rid in enumerate(record_set_ids):
    print(f"\nRecordSet/Distribution {idx+1}: @id = {rid}")
    try:
        # Try to get field information
        recordset = None
        if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
            for rs in dataset.metadata.recordSet:
                rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None)
                if rs_id == rid:
                    recordset = rs
                    break

        if recordset and hasattr(recordset, 'field') and recordset.field:
            print("  Fields:")
            for f in recordset.field:
                # Field could be dict or Str
                if isinstance(f, dict) and '@id' in f:
                    print(f"    - {f.get('name', '[no name]')} (@id: {f['@id']})")
                else:
                    print(f"    - {f}")
        else:
            print("  No field information present (likely file-based records)")
    except Exception as e:
        print(f"  Unable to examine record set {rid}: {e}")


## 3. Data Extraction

The main table with experimental records is stored in the FileObject whose `@id` is likely `'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'` according to the schema. We'll try to load records from it with `mlcroissant`.

In [ ]:
main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
record_sets = [main_record_set_id]
dataframes = {}

for record_set in record_sets:
    try:
        print(f"Loading records for RecordSet/FileObject with @id={record_set}")
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded {len(df)} rows with columns:")
        print(df.columns.tolist())
        display_cols = df.columns[:10] if len(df.columns) > 10 else df.columns
        display(df[display_cols].head())
    except Exception as e:
        print(f"Could not load record set {record_set}: {e}")

## 4. Exploratory Data Analysis (EDA)

Let us process the experimental table DataFrame. For this example, we'll select the field/column corresponding to normalized protein abundance, which commonly appears as `'Abundance_(Normalized)'` or similar. We'll reference its column by name (or by Croissant field `@id` if known). Adjust these names as necessary by examining the loaded DataFrame columns above.

In [ ]:
# Choose numeric field by column name (edit as per actual loaded columns)
main_df = dataframes[main_record_set_id]

# Replace this with the actual numeric column from your data
numeric_field = None
for c in main_df.columns:
    # Try to match a meaningful abundance field
    if "abundance" in c.lower() and "norm" in c.lower():
        numeric_field = c
        break
# Fallback: Try common alternatives
if not numeric_field:
    for c in main_df.columns:
        if c.lower().startswith("abundance") or "intensity" in c.lower():
            numeric_field = c
            break

assert numeric_field is not None, "Could not identify a numeric field for abundance in the data. Check columns in previous output."
print(f"Selected numeric field for filtering: {numeric_field}")

# Remove non-numeric values for safe calculations
main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')
threshold = main_df[numeric_field].quantile(0.9)  # Filter to top 10% by abundance
filtered_df = main_df[main_df[numeric_field] > threshold].copy()
print(f"Filtered records in {numeric_field} > {threshold:.2f} (top 10%): {len(filtered_df)} rows")
display(filtered_df[[numeric_field]].head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print("After normalization:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical variable, e.g. 'Sample' or 'Accession'
group_field = None
for possibility in ["Sample", "sample", "accession", "Accession", "Gene", "gene"]:
    if possibility in filtered_df.columns:
        group_field = possibility
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped by '{group_field}', mean abundance:")
    display(grouped_df.head())
else:
    print("No suitable group_field found for grouping. Columns:", filtered_df.columns.tolist())

## 5. Visualization

Visualize the distribution of normalized abundance and relation to a grouping variable (if present). We'll generate a histogram and, if a `group_field` was found, a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field], kde=True, bins=30)
plt.title(f"Distribution of {numeric_field} (top 10% records)")
plt.xlabel(numeric_field)
plt.show()

if group_field and filtered_df[group_field].nunique() < 40:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()
else:
    print("Skip groupwise boxplot (too many groups or group_field missing)")

## 6. Conclusion

In this notebook, we illustrated loading and exploring a Croissant dataset (`Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells`) using the `mlcroissant` library:
- Fetched metadata, inspected available record sets and fields using `@id`
- Loaded the main protein abundance table into a DataFrame
- Filtered for high-abundance records, normalized, and grouped data by key attributes
- Visualized data distributions

**Next steps** could include advanced analyses such as clustering, protein function annotation, or integrating with UniProt metadata as suggested by the schema and `dataPreprocessingProtocol`.

— This reproducible workflow enables transparent, FAIR-aligned machine learning and scientific analysis over metadata-rich data packages with [`mlcroissant`](https://mlcommons.org/croissant/).